<a href="https://colab.research.google.com/github/prithivika2007/PRODIGY_GA_01/blob/main/Task01_TextGeneration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets torch

In [4]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from torch.utils.data import Dataset

# Custom Dataset class (replaces TextDataset)
class TextDataset(Dataset):
    def __init__(self, tokenizer, file_path, block_size=128):
        with open(file_path, "r") as f:
            text = f.read()
        tokenized = tokenizer(text, return_tensors="pt", truncation=False)["input_ids"][0]
        self.examples = [tokenized[i:i+block_size] for i in range(0, len(tokenized) - block_size, block_size)]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return {"input_ids": self.examples[i], "labels": self.examples[i]}

# Load GPT-2
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained("gpt2")

# Load dataset
train_dataset = TextDataset(tokenizer, "data.txt", block_size=128)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training settings
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    logging_steps=100,
)
# Train
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

trainer.train()
print("✅ Training done!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training done!


In [8]:
from transformers import pipeline

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)

result = generator(
    "The secret to success is",
    max_new_tokens=80,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.9,
    pad_token_id=tokenizer.eos_token_id
)

print(result[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample', 'num_return_sequences', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The secret to success is to change the way people's lives," he said in a speech at the 2014 Conservative Alliance conference.

"We need to do something about this. One of all things, it will be that we create the world the way we want it to see it to be created, not something that we make we want - it when we want it, not what we want."


He said


In [9]:
prompts = [
    "Hard work and dedication will",
    "Believe in yourself because"
]

for p in prompts:
    result = generator(
        p,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    print(f"Prompt: {p}")
    print(result[0]['generated_text'])
    print("---")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'pad_token_id', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Hard work and dedication will
Hard work and dedication will be rewarded.


Don't get caught up in the hype. Stay honest and keep up the efforts and learn from the mistakes.

Keep up the efforts and grow, but don't get hung up over it.

And don't get hung up on it.


The goal is to make a difference in yourself.

Learn something that inspires you and is always
---
Prompt: Believe in yourself because
Believe in yourself because you're a true leader. Take it easy and be honest.

If you can't take it easy, don't go easy. Don't let success stand in the way of your dreams.

What do, however, do you do. You must have success. It is more than you can do.

It doesn't matter what your goals are, or who can be,
---
